In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    trim,
    lit,
    monotonically_increasing_id,
)
from functools import reduce

spark = SparkSession.builder.appName("normalize_to_silver").getOrCreate()

# <<< AQUÍ VA LA CONFIG DE BIGQUERY >>>
PROJECT_ID = "grupo2-essalud"
PLATA_DATASET = "essalud_plata"

# Bucket temporal que usará el conector para escribir en BigQuery
spark.conf.set("temporaryGcsBucket", "grupo2-essalud-datalake")

# ---------------------------
# CONFIG
# ---------------------------
silver_bucket = "gs://grupo2-essalud-datalake/plata"

# Tablas BRONCE en BigQuery
files = {
    "diabetes": "grupo2-essalud.essalud_bronce.Diabetes",
    "obesidad": "grupo2-essalud.essalud_bronce.Obesidad",
    "hipertension": "grupo2-essalud.essalud_bronce.Hipertension",
}

ubigeo_table = "grupo2-essalud.essalud_bronce.Ubigeo"

# Para guardar métricas de calidad
metricas = []   # cada elemento será un dict por tabla


# ---------------------------
# FUNCIÓN AUXILIAR: LIMPIEZA + MÉTRICAS
# ---------------------------
def limpiar_y_contar(df, nombre_tabla, columnas_clave):
    """
    - Cuenta total inicial
    - Cuenta registros con nulos en columnas_clave
    - Elimina nulos en columnas_clave
    - Elimina duplicados por columnas_clave
    - Devuelve df_limpio + métricas
    """

    # Total inicial
    total_inicial = df.count()

    # Nulos en columnas clave
    if columnas_clave:
        condicion_nulos = reduce(
            lambda acc, c: acc | col(c).isNull(),
            columnas_clave[1:],
            col(columnas_clave[0]).isNull()
        )
        nulos_eliminados = df.filter(condicion_nulos).count()
        df_sin_nulos = df.dropna(subset=columnas_clave)
    else:
        nulos_eliminados = 0
        df_sin_nulos = df

    # Duplicados
    if columnas_clave:
        antes_dups = df_sin_nulos.count()
        df_final = df_sin_nulos.dropDuplicates(columnas_clave)
        despues_dups = df_final.count()
        duplicados_eliminados = antes_dups - despues_dups
    else:
        df_final = df_sin_nulos
        duplicados_eliminados = 0

    # Guardar métricas en la lista global
    metricas.append({
        "tabla": nombre_tabla,
        "total_inicial": total_inicial,
        "nulos_eliminados": nulos_eliminados,
        "duplicados_eliminados": duplicados_eliminados,
        "total_final": df_final.count(),
    })

    print(
        f"[{nombre_tabla}] total_inicial={total_inicial}, "
        f"nulos_eliminados={nulos_eliminados}, "
        f"duplicados_eliminados={duplicados_eliminados}, "
        f"total_final={df_final.count()}"
    )

    return df_final


# ---------------------------
# LECTURA DE LAS 3 TABLAS BRONCE (BIGQUERY)
# ---------------------------
dfs = []
for enfermedad, table in files.items():
    df = (
        spark.read.format("bigquery")
        .option("table", table)
        .load()
    )

    # Normalizar strings (solo columnas string)
    tipos = dict(df.dtypes)
    for c in df.columns:
        if tipos[c] == "string":
            df = df.withColumn(c, trim(col(c)))

    # Añadir columna grupo_enfermedad según archivo
    if enfermedad == "diabetes":
        df = df.withColumn("grupo_enfermedad", lit("E1"))
    elif enfermedad == "obesidad":
        df = df.withColumn("grupo_enfermedad", lit("E2"))
    elif enfermedad == "hipertension":
        df = df.withColumn("grupo_enfermedad", lit("E3"))

    dfs.append(df)

df_all = (
    dfs[0]
    .unionByName(dfs[1], allowMissingColumns=True)
    .unionByName(dfs[2], allowMissingColumns=True)
)

# ---------------------------
# LECTURA UBIGEO DESDE BIGQUERY
# ---------------------------
ubigeo = (
    spark.read.format("bigquery")
    .option("table", ubigeo_table)
    .load()
)

ubigeo = ubigeo.select(
    col("Ubigeo").alias("ubigeo"),
    col("Departamento").alias("departamento"),
    col("Provincia").alias("provincia"),
    col("Distrito").alias("distrito"),
    col("Poblacion").alias("poblacion"),
)

# ---------------------------
# TABLA PACIENTE
# ---------------------------
paciente_raw = df_all.select(
    col("ID_PACIENTE").alias("id_paciente"),
    col("EDAD_PACIENTE").alias("edad_paciente"),
    col("SEXO_PACIENTE").alias("sexo_paciente"),
)

paciente = limpiar_y_contar(
    paciente_raw,
    "paciente",
    ["id_paciente"]
)

paciente.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{silver_bucket}/paciente")

# Tabla en BigQuery (Plata)
(
    paciente.write
    .format("bigquery")
    .option("table", f"{PROJECT_ID}.{PLATA_DATASET}.paciente")
    .mode("overwrite")
    .save()
)

# ---------------------------
# TABLA MEDICO
# ---------------------------
medico_raw = df_all.select(
    col("ID_MEDICO").alias("id_medico"),
    col("EDAD_MEDICO").alias("edad"),
)

medico = limpiar_y_contar(
    medico_raw,
    "medico",
    ["id_medico"]
)

medico.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{silver_bucket}/medico")

# Tabla en BigQuery (Plata)

(
    medico.write
    .format("bigquery")
    .option("table", f"{PROJECT_ID}.{PLATA_DATASET}.medico")
    .mode("overwrite")
    .save()
)

# ---------------------------
# TABLA CIE10
# ---------------------------
cie10_raw = df_all.select(
    col("COD_DIAG").alias("cod_enfermedad"),
    col("DIAGNOSTICO").alias("des_enfermedad"),
    col("grupo_enfermedad").alias("grupo_enfermedad"),
)

cie10 = limpiar_y_contar(
    cie10_raw,
    "cie10",
    ["cod_enfermedad"]
)

cie10.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{silver_bucket}/cie10")

# Tabla en BigQuery (Plata)

(
    cie10.write
    .format("bigquery")
    .option("table", f"{PROJECT_ID}.{PLATA_DATASET}.cie10")
    .mode("overwrite")
    .save()
)

# ---------------------------
# TABLA UBIGEO (PLATA)
# ---------------------------
ubigeo_plata = limpiar_y_contar(
    ubigeo,
    "ubigeo",
    ["ubigeo"]
)

ubigeo_plata.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{silver_bucket}/ubigeo")

# Tabla en BigQuery (Plata)

(
    ubigeo_plata.write
    .format("bigquery")
    .option("table", f"{PROJECT_ID}.{PLATA_DATASET}.ubigeo")
    .mode("overwrite")
    .save()
)


# ---------------------------
# TABLA DIAGNOSTICO (HECHOS)
# ---------------------------
diagnostico_raw = df_all.select(
    col("COD_DIAG").alias("cod_enfermedad"),
    col("ID_PACIENTE").alias("id_paciente"),
    col("UBIGEO").alias("ubigeo"),
    col("ID_MEDICO").alias("id_medico"),
    col("SERVICIO_HOSPITALARIO").alias("servicio_hospitalario"),
    col("ACTIVIDAD_HOSPITALARIA").alias("actividad_hospitalaria"),
    col("FECHA_MUESTRA").alias("fecha_muestra"),
)

# Claves mínimas para no dejar registros “huérfanos”
diagnostico_clean = limpiar_y_contar(
    diagnostico_raw,
    "diagnostico",
    ["cod_enfermedad", "id_paciente", "ubigeo"]
)

diagnostico = diagnostico_clean.withColumn(
    "diagnostico_id", monotonically_increasing_id()
)

diagnostico.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{silver_bucket}/diagnostico")

# Tabla en BigQuery (Plata)

(
    diagnostico.write
    .format("bigquery")
    .option("table", f"{PROJECT_ID}.{PLATA_DATASET}.diagnostico")
    .mode("overwrite")
    .save()
)

# ---------------------------
# TABLA PROCEDIMIENTO
# ---------------------------
proc1 = df_all.select(col("PROCEDIMIENTO_1").alias("cod_procedimiento")).where(
    col("PROCEDIMIENTO_1").isNotNull()
)

proc2 = df_all.select(col("PROCEDIMIENTO_2").alias("cod_procedimiento")).where(
    col("PROCEDIMIENTO_2").isNotNull()
)

procedimiento_raw = proc1.union(proc2)

procedimiento_clean = limpiar_y_contar(
    procedimiento_raw,
    "procedimiento",
    ["cod_procedimiento"]
)

procedimiento = procedimiento_clean.withColumn(
    "procedimiento_id", monotonically_increasing_id()
)

procedimiento.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{silver_bucket}/procedimiento")

# Tabla en BigQuery (Plata)

(
    procedimiento.write
    .format("bigquery")
    .option("table", f"{PROJECT_ID}.{PLATA_DATASET}.procedimiento")
    .mode("overwrite")
    .save()
)


# ---------------------------
# TABLA RESULTADO_PROCEDIMIENTO (HECHOS)
# ---------------------------
res1 = df_all.select(
    col("COD_DIAG").alias("cod_diagnostico"),
    col("PROCEDIMIENTO_1").alias("cod_procedimiento"),
    col("RESULTADO_1").alias("resultado"),
    col("UNIDADES_1").alias("unidades"),
    col("FEC_RESULTADO_1").alias("fecha_resultado"),
)

res2 = df_all.select(
    col("COD_DIAG").alias("cod_diagnostico"),
    col("PROCEDIMIENTO_2").alias("cod_procedimiento"),
    col("RESULTADO_2").alias("resultado"),
    col("UNIDADES_2").alias("unidades"),
    col("FEC_RESULTADO_2").alias("fecha_resultado"),
)

resultado_raw = res1.union(res2) \
    .where(col("cod_procedimiento").isNotNull())

resultado_clean = limpiar_y_contar(
    resultado_raw,
    "resultado_procedimiento",
    ["cod_diagnostico", "cod_procedimiento", "fecha_resultado"]
)

resultado_proc = resultado_clean.withColumn(
    "resultado_id", monotonically_increasing_id()
)

resultado_proc.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{silver_bucket}/resultado_procedimiento")

# Tabla en BigQuery (Plata)
(
    resultado_proc.write
    .format("bigquery")
    .option("table", f"{PROJECT_ID}.{PLATA_DATASET}.resultado_procedimiento")
    .mode("overwrite")
    .save()
)

# ---------------------------
# GUARDAR MÉTRICAS DE CALIDAD EN CSV (PLATA)
# ---------------------------
metricas_df = spark.createDataFrame(metricas)

metricas_df.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{silver_bucket}/metricas_calidad")

spark.stop()


25/11/28 22:32:08 INFO SparkEnv: Registering MapOutputTracker
25/11/28 22:32:09 INFO SparkEnv: Registering BlockManagerMaster
25/11/28 22:32:09 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
25/11/28 22:32:09 INFO SparkEnv: Registering OutputCommitCoordinator


[paciente] total_inicial=1308325, nulos_eliminados=0, duplicados_eliminados=666000, total_final=642325


25/11/28 22:33:23 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[medico] total_inicial=1308325, nulos_eliminados=0, duplicados_eliminados=1301635, total_final=6690


[cie10] total_inicial=1308325, nulos_eliminados=0, duplicados_eliminados=1308244, total_final=81


[ubigeo] total_inicial=1874, nulos_eliminados=0, duplicados_eliminados=0, total_final=1874


[diagnostico] total_inicial=1308325, nulos_eliminados=0, duplicados_eliminados=397610, total_final=910715


[procedimiento] total_inicial=2616650, nulos_eliminados=0, duplicados_eliminados=2616646, total_final=4


[resultado_procedimiento] total_inicial=2616650, nulos_eliminados=0, duplicados_eliminados=2544415, total_final=72235
